In [1]:
!pip install -q -U transformers peft datasets accelerate bitsandbytes
!pip uninstall -y torchao

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
import os, glob, json, re, gc, torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
path = glob.glob("/kaggle/input/**/eval_v1_scored.json", recursive=True)[0]
results = json.load(open(path))
print(len(results), results[0].keys())

40 dict_keys(['q', 'base', 'base_prompted', 'finetuned', 'score_base', 'score_base_prompted', 'score_finetuned'])


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32, device_map={"": 0})
model = PeftModel.from_pretrained(base, "Meow07/qwen2.5-0.5b-format-lora-v2").eval()

def gen(q):
    text = tok.apply_chat_template([{"role": "user", "content": q}],
                                   add_generation_prompt=True, tokenize=False)
    inp = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=300, do_sample=False,
                             repetition_penalty=1.05, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

for i, r in enumerate(results):
    r["finetuned_v2"] = gen(r["q"]); print(i, end=" ")

def ok(a):
    return bool(re.match(r"^Summary:.*\nKey points:\n- .*\n- .*\n- .*\nNext step:", a.strip(), re.S))
for name in ["finetuned", "finetuned_v2"]:
    print(name, "strict format:", sum(ok(r[name]) for r in results), "/", len(results))
json.dump(results, open("/kaggle/working/eval_v2_answers.json", "w"))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

... (3 outputs left out)

adapter_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 35.2MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 finetuned strict format: 37 / 40
finetuned_v2 strict format: 40 / 40


In [4]:
del model, base
gc.collect(); torch.cuda.empty_cache()
from transformers import BitsAndBytesConfig
JID = "Qwen/Qwen2.5-7B-Instruct"
jtok = AutoTokenizer.from_pretrained(JID)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
judge = AutoModelForCausalLM.from_pretrained(JID, quantization_config=bnb, device_map={"": 0})

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [5]:
import random, statistics as st
RUBRIC = """You are a strict grader. Rate the ANSWER to the QUESTION from 1 to 5.
5 = accurate, relevant and directly useful. 3 = partly useful or partly wrong.
1 = wrong, off-topic or useless.
Judge correctness and relevance only. Do NOT reward length or formatting.
Reply with only the digit."""

def ask_judge(system, user, n=3):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text = jtok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inp = jtok(text, return_tensors="pt").to(judge.device)
    with torch.no_grad():
        out = judge.generate(**inp, max_new_tokens=n, do_sample=False)
    return jtok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

def score(q, a):
    s = ask_judge(RUBRIC, f"QUESTION:\n{q}\n\nANSWER:\n{a}")
    m = re.search(r"[1-5]", s); return int(m.group()) if m else None

PAIR = """You compare two answers (A and B) to the same question.
Pick the one that is more accurate, relevant and useful. Ignore length and formatting.
Reply with only the letter A or B."""

random.seed(3)
wins = {"v1": 0, "v2": 0}
for r in results:
    r["score_finetuned_v2"] = score(r["q"], r["finetuned_v2"])
    flip = random.random() < 0.5                     # randomise position to avoid order bias
    a, b = (r["finetuned_v2"], r["finetuned"]) if flip else (r["finetuned"], r["finetuned_v2"])
    s = ask_judge(PAIR, f"QUESTION:\n{r['q']}\n\nANSWER A:\n{a}\n\nANSWER B:\n{b}", n=2)
    pick = "A" if "A" in s else "B"
    winner = ("v2" if pick == "A" else "v1") if flip else ("v1" if pick == "A" else "v2")
    r["pair_winner"] = winner; wins[winner] += 1

for name in ["base", "finetuned", "finetuned_v2"]:
    vals = [r["score_" + name] for r in results if r.get("score_" + name)]
    print(name, "mean quality:", round(st.mean(vals), 2), "n =", len(vals))
print("head-to-head wins:", wins)
json.dump(results, open("/kaggle/working/eval_v2_scored.json", "w"))

base mean quality: 3.5 n = 40
finetuned mean quality: 3.48 n = 40
finetuned_v2 mean quality: 3.58 n = 40
head-to-head wins: {'v1': 27, 'v2': 13}


In [6]:
d = sorted(results, key=lambda r: r["score_finetuned_v2"] - r["score_finetuned"])
for r in d[:2] + d[-2:]:
    print("\nQ:", r["q"][:140])
    print("v1 (", r["score_finetuned"], "):", r["finetuned"][:300])
    print("v2 (", r["score_finetuned_v2"], "):", r["finetuned_v2"][:300])


Q: What are 5 things I can do when it's raining in London? I am visiting for the first time and only for a week with my husband. We love to wal
v1 ( 4 ): Summary: When it rains heavily in London, consider exploring the city's parks, taking a stroll through historic neighborhoods, trying local cuisine, and enjoying outdoor activities like cycling or picnicking.
Key points:
- Explore parks like Hyde Park or St. James's Park.
- Visit historical landmark
v2 ( 3 ): Summary: When it rains in London, you can enjoy walking, eating delicious food, and exploring local attractions.
Key points:
- Walk through historic neighborhoods like St. James's Park or Trafalgar Square
- Enjoy a cup of tea at a cozy coffee shop
- Visit the British Museum for a museum visit
Next s

Q: What other strange and obscure music genres can I check out?
v1 ( 4 ): Summary: Explore various unique music genres such as Afrobeat, Hip-Hop Rock, House Music, and Reggae.
Key points:
- Afrobeat combines elements of African rhyt

In [7]:
v1w = v2w = incons = a_picks = 0
for r in results:
    s1 = ask_judge(PAIR, f"QUESTION:\n{r['q']}\n\nANSWER A:\n{r['finetuned']}\n\nANSWER B:\n{r['finetuned_v2']}", n=2)
    s2 = ask_judge(PAIR, f"QUESTION:\n{r['q']}\n\nANSWER A:\n{r['finetuned_v2']}\n\nANSWER B:\n{r['finetuned']}", n=2)
    a1 = s1.strip().upper().startswith("A"); a2 = s2.strip().upper().startswith("A")
    a_picks += a1 + a2
    w1 = "v1" if a1 else "v2"          # order 1: A is v1
    w2 = "v2" if a2 else "v1"          # order 2: A is v2
    if w1 != w2: incons += 1
    elif w1 == "v1": v1w += 1
    else: v2w += 1
print("consistent verdicts -> v1:", v1w, "| v2:", v2w, "| flipped with order:", incons)
print("picked position A in", a_picks, "of", 2 * len(results), "calls")

consistent verdicts -> v1: 15 | v2: 10 | flipped with order: 15
picked position A in 41 of 80 calls


In [8]:
import random
random.seed(11)
sample = random.sample(results, 10)
key = []
for i, r in enumerate(sample, 1):
    flip = random.random() < 0.5
    x, y = (r["finetuned_v2"], r["finetuned"]) if flip else (r["finetuned"], r["finetuned_v2"])
    key.append("v2" if flip else "v1")          # key[i-1] = which model is X
    print(f"\n#{i}  Q: {r['q'][:200]}\n--- X:\n{x[:450]}\n--- Y:\n{y[:450]}")


#1  Q: Can you give me some gift ideas for a 5-year-old boy that likes to play outside?
--- X:
Summary: Gift ideas for a 5-year-old boy who enjoys playing outside include fun toys like soccer balls, frisbees, and jump ropes; outdoor games such as tag or hide-and-seek; and books that feature adventure stories.
Key points:
- Soccer Balls
- Frisbees
- Jump Ropes
Next step: Research local parks or playgrounds where your child can play freely without supervision.
--- Y:
Summary: Gift ideas for a 5-year-old boy interested in playing outside include sports equipment, outdoor games, and educational toys.
Key points:
- Sports equipment like soccer balls or frisbees
- Outdoor games such as tag or hide-and-seek
- Educational toys like coloring books or science kits
Next step: Consider your child's interests when choosing gifts to ensure they are suitable.

#2  Q: Where can I go in Minneapolis to see art about different cultures? I'd also like to see modern art.
--- X:
Summary: Minneapolis offers

In [9]:
my_picks = "XYYXXYXYYX"      # replace with your 10 picks
wins = {"v1": 0, "v2": 0}
for p, k in zip(my_picks.upper(), key):
    other = "v1" if k == "v2" else "v2"
    wins[k if p == "X" else other] += 1
print("my verdict:", wins)

my verdict: {'v1': 6, 'v2': 4}
